# Nguyễn Thành Trí - MSSV: 6451071079
## Bài 2 (Trang 15-17) - Câu 8: Tạo ra các thuộc tính mới
1. Thuộc tính `age_class`: tích của tuổi (`age`) và loại vé (`pclass`).
2. Thuộc tính `fare_per_person`: giá vé chia cho số người đi cùng có làm trơn + 1 (`fare / (relatives + 1)`).
3. Hiển thị 10 dòng đầu tiên và thông tin cấu trúc dữ liệu sau khi hoàn tất tiền xử lý và biến đổi.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Bài 2 (15-17) - Câu 8: Tạo ra các thuộc tính mới

import os
import re
import numpy as np
import pandas as pd

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

# --- NẠP DỮ LIỆU & ÁP DỤNG CÁC BƯỚC TIỀN XỬ LÝ, BIẾN ĐỔI ---
def get_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join('..', rel_path),
        os.path.join('..', '..', rel_path),
        os.path.join('datasets', os.path.basename(rel_path)),
        os.path.join('..', 'datasets', os.path.basename(rel_path)),
        os.path.join('..', '..', 'datasets', os.path.basename(rel_path))
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Không tìm thấy file dữ liệu: {rel_path}')

csv_path = get_data_path('titanic.csv')
df = pd.read_csv(csv_path, na_values=['?'])
df.columns = df.columns.str.lower()
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['fare'] = pd.to_numeric(df['fare'], errors='coerce')

# Thuộc tính người thân (Câu 5)
df['relatives'] = df['sibsp'] + df['parch']
df.loc[df['relatives'] > 0, 'not_alone'] = 0
df.loc[df['relatives'] == 0, 'not_alone'] = 1
df['not_alone'] = df['not_alone'].astype(int)

# Tiền xử lý (Câu 6)
deck_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
df['cabin'] = df['cabin'].fillna("U0")
df['deck'] = df['cabin'].map(lambda x: re.compile(r"([a-zA-Z]+)").search(x).group())
df['deck'] = df['deck'].map(deck_map).fillna(0).astype(int)
df = df.drop(['cabin', 'ticket', 'boat', 'body', 'home.dest'], axis=1, errors='ignore')

mean, std = df["age"].mean(), df["age"].std()
is_null = df["age"].isnull().sum()
np.random.seed(42)
rand_age = np.random.randint(mean - std, mean + std, size=is_null)
age_slice = df["age"].copy()
age_slice[np.isnan(age_slice)] = rand_age
df["age"] = age_slice.astype(int)
df['embarked'] = df['embarked'].fillna('S')
df['fare'] = df['fare'].fillna(0).astype(int)

# Biến đổi thuộc tính (Câu 7)
titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df['title'] = df['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['title'] = df['title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                                   'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['title'] = df['title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
df['title'] = df['title'].map(titles).fillna(0).astype(int)
df = df.drop(['name'], axis=1)

df['sex'] = df['sex'].map({"male": 0, "female": 1})
df['embarked'] = df['embarked'].map({"S": 0, "C": 1, "Q": 2})

df.loc[df['age'] <= 11, 'age'] = 0
df.loc[(df['age'] > 11) & (df['age'] <= 18), 'age'] = 1
df.loc[(df['age'] > 18) & (df['age'] <= 22), 'age'] = 2
df.loc[(df['age'] > 22) & (df['age'] <= 27), 'age'] = 3
df.loc[(df['age'] > 27) & (df['age'] <= 33), 'age'] = 4
df.loc[(df['age'] > 33) & (df['age'] <= 40), 'age'] = 5
df.loc[(df['age'] > 40) & (df['age'] <= 66), 'age'] = 6
df.loc[df['age'] > 66, 'age'] = 7

df.loc[df['fare'] <= 7.91, 'fare'] = 0
df.loc[(df['fare'] > 7.91) & (df['fare'] <= 14.454), 'fare'] = 1
df.loc[(df['fare'] > 14.454) & (df['fare'] <= 31), 'fare'] = 2
df.loc[(df['fare'] > 31) & (df['fare'] <= 99), 'fare'] = 3
df.loc[(df['fare'] > 99) & (df['fare'] <= 250), 'fare'] = 4
df.loc[df['fare'] > 250, 'fare'] = 5
df['fare'] = df['fare'].astype(int)

# --- TẠO THUỘC TÍNH MỚI (CÂU 8) ---

# 1. Thuộc tính tuổi * loại vé (age_class)
df['age_class'] = df['age'] * df['pclass']

# 2. Thuộc tính giá vé trên mỗi người (fare_per_person)
df['fare_per_person'] = (df['fare'] / (df['relatives'] + 1)).astype(int)

# Hiển thị 10 dòng đầu tiên
print("\n--- 10 DÒNG ĐẦU TIÊN CỦA BẢNG DỮ LIỆU HOÀN THIỆN ---")
display(df.head(10))

# Thông tin cấu trúc dữ liệu cuối cùng
print("\n--- THÔNG TIN CẤU TRÚC DỮ LIỆU CUỐI CÙNG (df.info()) ---")
df.info()

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079

--- 10 DÒNG ĐẦU TIÊN CỦA BẢNG DỮ LIỆU HOÀN THIỆN ---


,pclass,survived,sex,age,sibsp,parch,fare,embarked,relatives,not_alone,deck,title,age_class,fare_per_person
0,1,1,1,4,0,0,4,0,0,1,2,2,4,4
1,1,1,0,0,1,2,4,0,3,0,3,4,0,1
2,1,0,1,0,1,2,4,0,3,0,3,2,0,1
3,1,0,0,4,1,2,4,0,3,0,3,1,4,1
4,1,0,1,3,1,2,4,0,3,0,3,3,3,1
5,1,1,0,6,0,0,2,0,0,1,5,1,6,2
6,1,1,1,6,1,0,3,0,1,0,4,2,6,1
7,1,0,0,5,0,0,0,0,0,1,1,1,5,0
8,1,1,1,6,2,0,3,0,2,0,3,3,6,1
9,1,0,0,7,0,0,3,1,0,1,8,1,7,3



--- THÔNG TIN CẤU TRÚC DỮ LIỆU CUỐI CÙNG (df.info()) ---
<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   pclass           1309 non-null   int64
 1   survived         1309 non-null   int64
 2   sex              1309 non-null   int64
 3   age              1309 non-null   int64
 4   sibsp            1309 non-null   int64
 5   parch            1309 non-null   int64
 6   fare             1309 non-null   int64
 7   embarked         1309 non-null   int64
 8   relatives        1309 non-null   int64
 9   not_alone        1309 non-null   int64
 10  deck             1309 non-null   int64
 11  title            1309 non-null   int64
 12  age_class        1309 non-null   int64
 13  fare_per_person  1309 non-null   int64
dtypes: int64(14)
memory usage: 143.3 KB
